<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/Tantalum_MLFF_FitSnap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Install LAMMPS and FitSNAP

In [ ]:
!python --version

In [ ]:
pip install jarvis-tools

If you are running locally and have already installed LAMMPS and FitSNAP, skip this step.

In [ ]:
# Install LAMMPS with Python interface.

!apt-get update
!apt install -y cmake build-essential git ccache openmpi-bin libopenmpi-dev python3.10-venv
!pip install --upgrade pip
!pip install numpy torch scipy virtualenv psutil pandas tabulate mpi4py Cython sklearn
!pip install ase
!pip install fitsnap3
%cd /content
!rm -rf lammps
!git clone https://github.com/lammps/lammps.git lammps
%cd /content/lammps
!rm -rf build
!mkdir build
%cd build
!cmake ../cmake -DLAMMPS_EXCEPTIONS=yes \
               -DBUILD_SHARED_LIBS=yes \
               -DMLIAP_ENABLE_PYTHON=yes \
               -DPKG_PYTHON=yes \
               -DPKG_ML-SNAP=yes \
               -DPKG_ML-IAP=yes \
               -DPKG_ML-PACE=yes \
               -DPKG_SPIN=yes \
               -DPYTHON_EXECUTABLE:FILEPATH=`which python`
!make -j 2
!make install-python

# Install FitSNAP.

%cd /content
!rm -rf FitSNAP
!git clone https://github.com/FitSNAP/FitSNAP
#!git clone -b hackathon https://github.com/rohskopf/FitSNAP

# Set environment variables.

!$PYTHONPATH
%env PYTHONPATH=/env/python:/bin/bash:
%env LD_LIBRARY_PATH=/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/content/lammps/build

# Move into FitSNAP directory
%cd FitSNAP

# Check if Python LAMMPS is working

In [ ]:
import lammps
lmp = lammps.lammps()
print(lmp)

# Scrape data with FitSNAP ASE scraper

In [ ]:
import numpy as np
from mpi4py import MPI
from fitsnap3lib.fitsnap import FitSnap
from fitsnap3lib.scrapers.ase_funcs import get_apre
from fitsnap3lib.scrapers.ase_funcs import ase_scraper
from fitsnap3lib.tools.group_tools import make_table
from ase.io import read

In [ ]:
# Create an input dictionary containing settings.

settings = \
{
"BISPECTRUM":
    {
    "numTypes": 1,
    "twojmax": 6,
    "rcutfac": 4.67637,
    "rfac0": 0.99363,
    "rmin0": 0.0,
    "wj": 1.0,
    "radelem": 0.5,
    "type": "Ta",
    "wselfallflag": 0,
    "chemflag": 0,
    "bzeroflag": 0,
    "quadraticflag": 0,
    },
"CALCULATOR":
    {
    "calculator": "LAMMPSSNAP",
    "energy": 1,
    "force": 1,
    "stress": 1
    },
"SOLVER":
    {
    "solver": "SVD"
    },
"OUTFILE":
    {
    "metrics": "Ta_metrics.md",
    "potential": "Ta_pot"
    },
"REFERENCE":
    {
    "units": "metal",
    "atom_style": "atomic",
    "pair_style": "hybrid/overlay zero 10.0 zbl 4.0 4.8",
    "pair_coeff1": "* * zero",
    "pair_coeff2": "* * zbl 73 73"
    }
}

In [ ]:
# Make a fitsnap instance.
fs = FitSnap(settings, arglist=["--overwrite"])

In [ ]:
# When dealing with groups, it is best to use the `group_table`.
# First make a dictionary of settings for each group, which contains:
# - a key called "group_sections" with a list that names the columns of the table.
# - keys of group names where each key contains a list of column data.

group_settings = {
    "group_sections": ["training_size", "testing_size", "eweight", "fweight", "vweight"],
    "Displaced_A15" :  [0.8,    0.2,      100,            1,               1.00E-05],
    "Displaced_BCC" :  [0.8,    0.2,       100,             1,             1.00E-05],
    "Displaced_FCC" :  [0.8,    0.2,       100,             1,             1.00E-05],
    "Elastic_BCC"   :  [0.8,    0.2,     1.00E-08,        1.00E-08,        1.00E-05],
    "Elastic_FCC"   :  [0.8,    0.2,     1.00E-09,        1.00E-09,        1.00E-05],
    "GSF_110"       :  [0.8,    0.2,      100,             1,              1.00E-05],
    "GSF_112"       :  [0.8,    0.2,      100,             1,              1.00E-05],
    "Liquid"        :  [0.8,    0.2,       4.67E+02,        1,             1.00E-05],
    "Surface"       :  [0.8,    0.2,       100,             1,             1.00E-05],
    "Volume_A15"    :  [0.8,    0.2,      1.00E+00,        1.00E-09,       1.00E-05],
    "Volume_BCC"    :  [0.8,    0.2,      1.00E+00,        1.00E-09,       1.00E-05],
    "Volume_FCC"    :  [0.8,    0.2,      1.00E+00,        1.00E-09,       1.00E-05]
    }

In [ ]:
group_table = make_table(group_settings)

In [ ]:
# Make ASE frames for each group; do this however you want, we simply read from filenames that share
# group names here.
for name in group_table:
    frames = read(f"examples/Ta_XYZ/XYZ/{name}.xyz", ":")
    group_table[name]["frames"] = frames
    group_table[name]["nconfigs"] = len(frames)

In [ ]:
# Inject group data into the fitsnap list of data dictionaries.
data = ase_scraper(group_table)
print(f"Found {len(data)} configurations")

In [ ]:
# Calculate descriptors for all configurations.
fs.process_configs(data)

# Perform a fit.
fs.solver.perform_fit()

# Analyze error metrics.
fs.solver.error_analysis()

In [ ]:
# Dataframe of detailed errors per group.
print(fs.solver.errors)

# Convert to JARVIS Atoms format

In [ ]:
from jarvis.core.atoms import Atoms as JAtoms
def fs_to_jatoms(fs_entry=[]):
  elements=fs_entry['AtomTypes']
  cart_coords=fs_entry['Positions']
  lattice_mat=fs_entry['Lattice']
  atoms=JAtoms(elements=elements,coords=cart_coords,lattice_mat=lattice_mat,cartesian=True)
  return atoms



In [ ]:
atms = fs_to_jatoms(data[0])

In [ ]:
atms

Convert to JARVIS Atoms from the FitSNAP `data` list.

In [ ]:
from jarvis.db.jsonutils import dumpjson
mem=[]
for ii,i in enumerate(data):
  ta_id='Ta_fit_'+str(ii)
  atms = fs_to_jatoms(i)
  info={}
  info['atoms']=atms.to_dict()
  info['id']=ta_id
  info['energy']=i['Energy']
  info['energy_per_atom']=i['Energy']/atms.num_atoms
  info['forces']=i['Forces'].tolist()
  info['stress']=i['Stress'].tolist()
  mem.append(info)



In [ ]:
dumpjson(data=mem,filename='ta_fitsnap.json')

In [ ]:
!ls